# World Cup 2026 Match Predictor — Backbone

Predict **Home Win / Draw / Away Win** for international matches.

This notebook builds the *skeleton* end-to-end so we have a running prediction machine, then we grow it by adding feature columns (born-abroad %, head-to-head, styles).

**Workflow (mirrors the housing notebook, but classification instead of regression):**
1. Load `results.csv` (every international match ever — our backbone).
2. **One forward pass in date order** computing, for each match, the *pre-kickoff* Elo, recent form, and goal averages of both teams. (Forward pass = no data leakage.)
3. Label each match W/D/L.
4. Filter to **modern football** (ratings warmed up on all history, but we only train on modern matches).
5. **Time-based split** (train on old, test on recent — never random for time series).
6. Baseline classifier → evaluate → predict a real 2026 matchup.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

print("python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)

pd.set_option("display.max_columns", 50)

## Config knobs

Everything you'll want to tune lives here.

- `MODERN_CUTOFF` — we compute Elo on ALL history (so ratings are warm/accurate) but only **train & evaluate** on matches from this year onward. Try 1986 / 1994 / 2002 later and see if accuracy improves.
- `TEST_FROM` — matches on/after this date are the held-out test set (time-based, not random).

In [ ]:
DATA_PATH   = Path("datasets/results.csv")   # <-- put the Kaggle file here
MODERN_CUTOFF = 1986   # train/eval only on matches from this year on
TEST_FROM     = "2022-01-01"  # held-out test set = matches on/after this date

# Elo parameters (tunable)
ELO_START = 1500      # every team's cold-start rating
ELO_K     = 40        # how fast ratings move after a result
HOME_ADV  = 65        # Elo points added to the home side (0 if neutral venue)
FORM_N    = 5         # how many recent matches define 'form'

## 1. Load the backbone data

**`results.csv`** = the file inside the Kaggle dataset
[martj42/international-football-results](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017?select=results.csv).

Download it and drop it at `datasets/results.csv`. Columns:
`date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`.

In [ ]:
def load_matches(path=DATA_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Couldn't find {path}. Download results.csv from the martj42 Kaggle "
            f"dataset and place it there."
        )
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.dropna(subset=["home_score", "away_score"])   # keep only played matches
    df = df.sort_values("date").reset_index(drop=True)     # date order = required for forward pass
    return df

matches = load_matches()
print(matches.shape)
matches.head()

: 

In [ ]:
matches.info()

## 2. Forward pass — compute pre-kickoff features (Elo + form + goals)

We walk through every match **in date order**. For each match we FIRST record what both teams looked like *before* kickoff (their current Elo, last-5 form, recent goal averages), THEN update those numbers using the result.

Recording *before* updating is what guarantees **no leakage** — a feature can never peek at the match it's trying to predict.

Same loop computes Elo, form, and goal averages at once, so one pass gives us all our backbone features.

In [ ]:
from collections import defaultdict, deque

elo      = defaultdict(lambda: ELO_START)          # team -> current Elo
recent   = defaultdict(lambda: deque(maxlen=FORM_N))  # team -> deque of recent (points, gf, ga)

def form_points(team):
    """Points earned in the last FORM_N matches (3 win / 1 draw / 0 loss)."""
    return sum(p for p, gf, ga in recent[team])

def avg_gf_ga(team):
    """Average goals scored / conceded over the last FORM_N matches."""
    if not recent[team]:
        return np.nan, np.nan
    gf = np.mean([g for p, g, a in recent[team]])
    ga = np.mean([a for p, g, a in recent[team]])
    return gf, ga

rows = []
for m in matches.itertuples(index=False):
    h, a = m.home_team, m.away_team
    neutral = bool(m.neutral)
    ha = 0 if neutral else HOME_ADV

    # --- RECORD pre-kickoff state (features) ---
    eh, ea = elo[h], elo[a]
    hgf, hga = avg_gf_ga(h)
    agf, aga = avg_gf_ga(a)
    rows.append({
        "date": m.date,
        "home_team": h, "away_team": a, "neutral": int(neutral),
        "elo_home": eh, "elo_away": ea,
        "elo_diff": (eh + ha) - ea,
        "form_home": form_points(h), "form_away": form_points(a),
        "form_diff": form_points(h) - form_points(a),
        "gf_home": hgf, "ga_home": hga, "gf_away": agf, "ga_away": aga,
        "home_score": m.home_score, "away_score": m.away_score,
    })

    # --- UPDATE state using the actual result ---
    exp_h = 1.0 / (1.0 + 10 ** ((ea - (eh + ha)) / 400.0))   # expected home score 0..1
    if   m.home_score > m.away_score: s_h, ph, pa = 1.0, 3, 0
    elif m.home_score < m.away_score: s_h, ph, pa = 0.0, 0, 3
    else:                             s_h, ph, pa = 0.5, 1, 1
    gd = abs(m.home_score - m.away_score)
    mult = np.log(gd + 1) + 1        # margin-of-victory multiplier
    change = ELO_K * mult * (s_h - exp_h)
    elo[h] += change
    elo[a] -= change

    recent[h].append((ph, m.home_score, m.away_score))
    recent[a].append((pa, m.away_score, m.home_score))

feat = pd.DataFrame(rows)
print(feat.shape)
feat.tail()

### Sanity check: do our computed Elo ratings look reasonable?

Top teams by our current (latest) Elo. Brazil / France / Argentina / Spain should sit near the top — if they do, the forward pass is working. (This is where you can cross-check against the downloaded Elo dataset.)

In [ ]:
top = pd.Series(dict(elo)).sort_values(ascending=False).head(20)
top

## 3. Label each match: Home win = 2, Draw = 1, Away win = 0

In [ ]:
feat["result"] = np.sign(feat["home_score"] - feat["away_score"]).astype(int) + 1  # -> 0/1/2
feat["result"].value_counts().rename({0: "away_win", 1: "draw", 2: "home_win"})

## 4. Filter to modern football

Note: Elo was computed on ALL history above, so ratings entering 1986 are already warmed up. Here we just choose which rows to *model on*. Also drop the earliest modern matches that still have no form history (NaN).

In [ ]:
modern = feat[feat["date"].dt.year >= MODERN_CUTOFF].copy()
modern = modern.dropna(subset=["gf_home", "gf_away"])   # need some form history
print(f"{len(modern)} matches from {MODERN_CUTOFF} onward")
modern.head()

## 5. Time-based train/test split

For time-series we NEVER split randomly — that would train on the future to predict the past. Train on older matches, test on recent ones.

In [ ]:
FEATURES = ["elo_diff", "form_diff", "neutral",
            "gf_home", "ga_home", "gf_away", "ga_away"]

train = modern[modern["date"] <  TEST_FROM]
test  = modern[modern["date"] >= TEST_FROM]

X_train, y_train = train[FEATURES], train["result"]
X_test,  y_test  = test[FEATURES],  test["result"]
print(f"train: {len(train)}   test: {len(test)}")

## 6. Baseline models + evaluate

Two references to beat:
- **Always predict home win** (the majority class) — the dumbest possible baseline.
- **Logistic regression** on our features.
- **Random forest** — usually the stronger of the two.

We report accuracy and log-loss (log-loss rewards good *probabilities*, which is what you want for a bracket).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report

# dumb baseline
majority = y_train.mode()[0]
print(f"Always-home-win accuracy: {(y_test == majority).mean():.3f}\n")

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
rf.fit(X_train, y_train)

for name, model in [("LogReg", logreg), ("RandomForest", rf)]:
    pred  = model.predict(X_test)
    proba = model.predict_proba(X_test)
    print(f"{name:12s}  acc={accuracy_score(y_test, pred):.3f}  "
          f"log_loss={log_loss(y_test, proba):.3f}")

In [ ]:
print(classification_report(y_test, rf.predict(X_test),
                            target_names=["away_win", "draw", "home_win"]))

## 7. Predict a real matchup

Feed in the two teams' *current* Elo and form to get W/D/L probabilities. This is the payoff — the same function will predict any 2026 fixture.

In [ ]:
def predict_match(home, away, model=rf, neutral=True):
    ha = 0 if neutral else HOME_ADV
    hgf, hga = avg_gf_ga(home)
    agf, aga = avg_gf_ga(away)
    x = pd.DataFrame([{
        "elo_diff": (elo[home] + ha) - elo[away],
        "form_diff": form_points(home) - form_points(away),
        "neutral": int(neutral),
        "gf_home": hgf, "ga_home": hga, "gf_away": agf, "ga_away": aga,
    }])[FEATURES]
    p = model.predict_proba(x)[0]
    return {f"{away}_win": p[0], "draw": p[1], f"{home}_win": p[2]}

# example — change to any two teams present in the data
predict_match("United States", "Mexico", neutral=True)

## Next steps — growing the backbone

The skeleton runs. Now add columns to `FEATURES` one at a time and re-run — nothing else changes:

1. **Head-to-head** — in the forward pass, also track results between each *pair* of teams; add `h2h_diff`.
2. **Born-abroad %** (luarai) — build a `team -> pct_born_abroad` table, merge on team, add `born_abroad_diff`.
3. **Style stats** (possession, xG, pass %) — merge from FBref, add diffs for the 'clash of styles'.
4. **Tune**: cross-validate with `TimeSeriesSplit`, try `MODERN_CUTOFF` = 1994/2002, calibrate probabilities with `CalibratedClassifierCV`.
5. **Evaluate once** on the most recent qualifiers, then predict the 2026 bracket.